# Step 1 — Load the Graph

Tool: pandas + NetworkX
Read both CSVs with pandas, then construct a NetworkX DiGraph. Each node carries (lat, lon) attributes; each edge stores its hourly travel-time vector (hour_0 through hour_23 or however the dataset is structured) plus any segment ID needed to query the random forest model.


In [2]:
import sys
print(sys.executable)

/Users/elainehuan/anaconda3/envs/traffic/bin/python


In [7]:
import pandas as pd
import numpy as np
import networkx as nx
import heapq
from sklearn.ensemble import RandomForestRegressor
from scipy.spatial import KDTree

# 1. LOAD DATA
nodes = pd.read_csv("../data/ml_datasets/routing_nodes.csv")
edges = pd.read_csv("../data/ml_datasets/routing_edges.csv")
ml_data = pd.read_csv("../data/ml_datasets/congestion_ml.csv")

# 2. TRAIN ML ORACLE (Using the large dataset for high accuracy)
# We predict travel time based on features known at the start (Time & Location)
ml_features = ['hour', 'is_weekend', 'is_rush_hour',
               'borough_Bronx', 'borough_Brooklyn', 'borough_Manhattan',
               'borough_Queens', 'borough_Staten Island']
ml_data_clean = ml_data.dropna(subset=['avg_travel_time'])
X_train = ml_data_clean[ml_features]
y_train = ml_data_clean['avg_travel_time']

rf_model = RandomForestRegressor(n_estimators=30, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# 3. BUILD & HEAL THE GRAPH
G = nx.DiGraph()
edge_lookup = {}

# Add physical intersections
for _, n in nodes.iterrows():
    G.add_node(n['node_id'], lat=n['lat'], lon=n['lon'])

# Add existing segments with "Safety Floors" to prevent 0-sec travel times
for _, row in edges.iterrows():
    u, v = row['from_node'], row['to_node']
    if u not in G or v not in G: continue
    if (u, v) not in edge_lookup:
        edge_lookup[(u, v)] = {'times': {}, 'borough': row['borough']}

    # Floor travel time at 10 seconds to prevent teleportation bugs
    t_val = row['est_travel_time_sec'] if row['est_travel_time_sec'] > 10 else 10
    edge_lookup[(u, v)]['times'][row['HH']] = t_val
    G.add_edge(u, v)

# HEALER: Create virtual links between nodes within 1.5km to ensure a connected city network
coords = nodes[['lat', 'lon']].values
tree = KDTree(coords)
pairs = tree.query_pairs(0.015) # Approx 1.6km radius

for i, j in pairs:
    u, v = nodes.iloc[i]['node_id'], nodes.iloc[j]['node_id']
    if not G.has_edge(u, v):
        dist = np.sqrt((nodes.iloc[i]['lat']-nodes.iloc[j]['lat'])**2 +
                       (nodes.iloc[i]['lon']-nodes.iloc[j]['lon'])**2) * 111000
        time_est = dist / 8.0 # Estimated 18mph for virtual links
        edge_lookup[(u, v)] = {'times': {}, 'borough': 'Manhattan', 'is_virtual': True, 'base_time': time_est}
        G.add_edge(u, v)
        edge_lookup[(v, u)] = {'times': {}, 'borough': 'Manhattan', 'is_virtual': True, 'base_time': time_est}
        G.add_edge(v, u)

# 4. DYNAMIC COST FUNCTION
def get_dynamic_cost(u, v, current_time_sec):
    edge = edge_lookup[(u, v)]
    hour = (int(current_time_sec) // 3600) % 24

    # 1. Historical Lookup
    if hour in edge['times']: return edge['times'][hour]
    # 2. Virtual/Distance Estimate
    if 'is_virtual' in edge: return edge['base_time']
    # 3. ML Oracle Prediction
    b = edge['borough']
    b_flags = {f'borough_{name}': (1 if name == b else 0) for name in ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']}
    inp = pd.DataFrame([{'hour': hour, 'is_weekend': 0, 'is_rush_hour': 1 if (7<=hour<=9 or 16<=hour<=19) else 0, **b_flags}])
    return rf_model.predict(inp[ml_features])[0]

# 5. TIME-DEPENDENT ROUTING ENGINE
def predict_route(source, target, start_hour):
    start_sec = start_hour * 3600
    pq = [(start_sec, source)]
    best_times = {source: start_sec}
    parent = {}

    while pq:
        t, u = heapq.heappop(pq)
        if u == target: break
        if t > best_times.get(u, float('inf')): continue

        for v in G.successors(u):
            cost = get_dynamic_cost(u, v, t)
            new_t = t + max(5, cost)
            if new_t < best_times.get(v, float('inf')):
                best_times[v] = new_t
                parent[v] = u
                heapq.heappush(pq, (new_t, v))

    if target not in parent: return None, 0
    path = []
    curr = target
    while curr != source:
        path.append(curr); curr = parent[curr]
    path.append(source)
    return path[::-1], best_times[target] - start_sec

# 6. TEST
#  find nodes in the main connected component to show it works
wccs = list(nx.weakly_connected_components(G))
main_net = max(wccs, key=len)
s_node, t_node = list(main_net)[0], list(main_net)[-1]

path, travel_time = predict_route(s_node, t_node, 8) # 8:00 AM Departure

print(f"Nodes in Connected Network: {len(main_net)}")
print(f"Routing from: {s_node}")
print(f"Routing to:   {t_node}")
print(f"Path Found:   {len(path)} nodes")
print(f"Travel Time:  {travel_time/60:.2f} minutes")

Nodes in Connected Network: 45
Routing from: BROOKLYN BRIDGE @ EAST RIVER SHORELINE
Routing to:   HAMILTON AVENUE @ MILL STREET
Path Found:   4 nodes
Travel Time:  9.78 minutes


In [8]:
print(nodes.shape)
print(edges.shape)

(222, 3)
(998, 24)


Step 6

In [11]:
import folium

def render_map(G, path):
    if path is None or len(path) == 0:
        return None

    coords = [(G.nodes[n]["lat"], G.nodes[n]["lon"]) for n in path]

    # center map at midpoint
    center_lat = sum(c[0] for c in coords) / len(coords)
    center_lon = sum(c[1] for c in coords) / len(coords)

    m = folium.Map(location=[center_lat, center_lon], zoom_start=13)

    # draw route
    folium.PolyLine(coords, color="blue", weight=5).add_to(m)

    # start/end markers
    folium.Marker(coords[0], tooltip="Start").add_to(m)
    folium.Marker(coords[-1], tooltip="End").add_to(m)

    return m

Step 7

In [ ]:
# This STREAMLIT APP should actually be a separate .py file, but for simplicity we also include the code here.
# RUN THE UI WITH: [streamlit run streamlit_UI.py] in terminal

import streamlit_UI as st
from streamlit_folium import st_folium

st.title("Predictive Traffic Routing")

node_ids = list(G.nodes)

start = st.selectbox("Start Node", node_ids)
end = st.selectbox("End Node", node_ids)
hour = st.slider("Departure Hour", 0, 23, 8)

if st.button("Find Route"):
    path, total_time = predict_route(start, end, hour)

    if path is None:
        st.error("No route found")
    else:
        st.write(f"Estimated travel time: {total_time / 60:.2f} minutes")

        m = render_map_with_congestion(G, path, hour)
        st_folium(m, width=700, height=500)

2026-05-03 02:16:50.180 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 02:16:50.182 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 02:16:50.203 
  command:

    streamlit run /Users/elainehuan/anaconda3/envs/traffic/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-05-03 02:16:50.203 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 02:16:50.203 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 02:16:50.204 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 02:16:50.204 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-05-03 02:16:50.204 Thread 'MainThread': missing ScriptRunContext! This warning c

Step 8

In [13]:
# test: static time
def static_route(source, target, hour):
    def weight(u, v, d):
        return get_dynamic_cost(u, v, hour * 3600)

    try:
        path = nx.shortest_path(G, source, target, weight=weight)
        total = sum(
            weight(path[i], path[i+1], None)
            for i in range(len(path)-1)
        )
        return path, total
    except:
        return None, None

In [14]:
# test: free flow time
def free_flow_cost(u, v):
    edge = edge_lookup[(u, v)]
    if edge['times']:
        return min(edge['times'].values())
    return edge.get('base_time', 60)

In [15]:
# EVALUATION LOOP
import random
import pandas as pd

def evaluate_model(n_samples=50):
    nodes_list = list(main_net)
    results = []

    for _ in range(n_samples):
        s, t = random.sample(nodes_list, 2)

        td_path, td_time = predict_route(s, t, 8)
        static_path, static_time = static_route(s, t, 8)

        if td_time and static_time:
            results.append({
                "td_time": td_time,
                "static_time": static_time,
                "improved": td_time < static_time
            })

    return pd.DataFrame(results)


# METRICS
df = evaluate_model(100)

print("Avg TD time:", df["td_time"].mean()/60)
print("Avg Static time:", df["static_time"].mean()/60)

print("TD better %:", df["improved"].mean())


# STABILITY
def stability_test(s, t):
    path1, _ = predict_route(s, t, 8)
    path2, _ = predict_route(s, t, 9)

    if path1 is None or path2 is None:
        return None

    overlap = len(set(path1) & set(path2)) / len(set(path1) | set(path2))
    return overlap

Avg TD time: 8.46065108941007
Avg Static time: 8.46065108941007
TD better %: 0.46464646464646464
